In [0]:
dbutils.widgets.text("catalog_name", "", "Catalog Name")
dbutils.widgets.text("entity", "", "Entity Name")
dbutils.widgets.dropdown("function", "upgrade", ["upgrade", "downgrade"], "Function")

In [0]:
catalog=dbutils.widgets.get("catalog_name")
entity = dbutils.widgets.get("entity")
function=dbutils.widgets.get("function")

In [0]:
"""
Migration: Create entity-specific testmigration table
Revision: 002_create_entity_testmigration
Down Revision: 001_create_test_migration
"""

def upgrade():
    """
    Create {entity}_unified_prod_testmigration table based on {entity}_unified_prod schema
    """
    print(f"Applying migration: Create entity-specific testmigration table")
    print(f"Entity: {entity}")
    print("-" * 80)
    
    # Source table name
    source_table = f"{catalog}.silver.{entity}_unified_prod"
    target_table = f"{catalog}.silver.{entity}_unified_prod_testmigration"
    
    # Get the source table schema
    source_df = spark.table(source_table)
    
    # Create target table with same schema as source
    print(f"Creating table: {target_table}")
    
    # Use CREATE TABLE LIKE to copy schema
    create_query = f"""
        CREATE TABLE IF NOT EXISTS {target_table}
        AS SELECT * FROM {source_table}
    """
    print(f"Create query: {create_query}")
    spark.sql(create_query)
    print(f"✓ Created table: {target_table}")
    
    print("-" * 80)
    print(f"✓ Migration applied successfully")


def downgrade():
    """
    Drop entity-specific testmigration table
    """
    print(f"Reverting migration: Create entity-specific testmigration table")
    print(f"Entity: {entity}")
    print("-" * 80)
    
    target_table = f"{catalog}.silver.{entity}_unified_prod_testmigration"
    
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    print(f"✓ Dropped table: {target_table}")
    
    print("-" * 80)
    print(f"✓ Migration reverted successfully")

In [0]:
if function == "downgrade":
    downgrade()
else:
    upgrade()